# Desafio Integrador – Pré-processamento de Dados para Machine Learning

**Curso:** Machine Learning Aplicada à Indústria  
**Desafio complementar às Aulas 1, 2 e 3**

## Objetivo
Aplicar os conceitos de:
- entendimento do problema;
- inspeção inicial da base;
- tratamento de faltantes;
- remoção de duplicatas;
- tratamento de outliers;
- normalização/padronização;
- codificação de variáveis categóricas;
- preparação final da base para modelagem.

## Entregáveis
1. Base diagnosticada.
2. Base tratada.
3. Base final pronta para ML.
4. Resumo técnico das decisões adotadas.

In [ ]:
# Instale as bibliotecas se necessário
# !pip install pandas numpy matplotlib seaborn scikit-learn kagglehub -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

pd.set_option('display.max_columns', None)

## 1. Carregamento da base

Você pode usar:
- a base das aulas anteriores; ou
- uma base do Kaggle baixada manualmente.

Substitua o caminho abaixo, se necessário.

In [ ]:
CAMINHO_ARQUIVO = '/content/dados_desafio_preprocessamento_industria_v2.csv'
df = pd.read_csv(CAMINHO_ARQUIVO)
df.head()

,timestamp,linha_producao,turno,produto_quimico,tipo_valvula,status_sensor,vazao_setpoint_l_min,pressao_linha_bar,temperatura_produto_c,abertura_valvula_pct,vibracao_bomba_mm_s,ph_produto,consumo_energia_kwh,tempo_resposta_seg,erro_vazao_pct,leituras_invalidas,horas_operacao_bomba,status_instrumentacao
0,2026-03-06 07:00:00,Linha_C,Manhã,Ácido_Y,Borboleta,OK,127.66,4.46,45.51,80.42,3.53,7.58,18.01,3.29,4.66,3,3133,Normal
1,2026-03-02 17:00:00,Linha_B,Noite,Base_Z,Esfera,OK,131.68,5.09,35.64,61.61,1.61,8.19,25.60,1.62,5.64,1,1547,Normal
2,2026-03-03 07:00:00,Linha_C,Tarde,Ácido_Y,Globo,Instável,158.94,5.06,36.43,45.48,4.01,8.02,30.05,1.45,5.06,2,2300,Alerta
3,2026-03-07 14:00:00,Linha_A,Noite,Solvente_X,Globo,OK,157.35,4.73,39.69,47.95,2.41,6.85,14.89,2.74,3.06,1,3694,Normal
4,2026-03-10 02:00:00,Linha_B,Noite,Base_Z,Esfera,OK,121.38,4.80,32.98,54.65,1.98,6.38,15.02,1.70,7.41,1,2337,Normal


## 2. Compreensão inicial da base

**Tarefa:**  
Analise a estrutura da base e descreva:
- quantidade de linhas e colunas;
- tipos de dados;
- variável-alvo provável;
- possíveis problemas de qualidade dos dados.

In [ ]:
# Complete a análise inicial
print("Dimensões:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

Dimensões: (235, 18)

Tipos de dados:
timestamp                 object
linha_producao            object
turno                     object
produto_quimico           object
tipo_valvula              object
status_sensor             object
vazao_setpoint_l_min     float64
pressao_linha_bar        float64
temperatura_produto_c    float64
abertura_valvula_pct     float64
vibracao_bomba_mm_s      float64
ph_produto               float64
consumo_energia_kwh      float64
tempo_resposta_seg       float64
erro_vazao_pct           float64
leituras_invalidas         int64
horas_operacao_bomba       int64
status_instrumentacao     object
dtype: object


In [ ]:
# Exiba estatísticas descritivas
df.select_dtypes(exclude = np.number).columns.to_list()

['timestamp',
 'linha_producao',
 'turno',
 'produto_quimico',
 'tipo_valvula',
 'status_sensor',
 'status_instrumentacao']

In [ ]:
# Defina a variável-alvo, se houver
target = 'status_instrumentacao'
target

'status_instrumentacao'

## 3. Diagnóstico da qualidade dos dados

**Tarefa:**  
Verifique:
- valores faltantes;
- duplicatas;
- colunas numéricas;
- colunas categóricas.

In [ ]:
# Conte os valores faltantes por coluna
df.isnull().sum()

,0
timestamp,0
linha_producao,0
turno,0
produto_quimico,0
tipo_valvula,10
status_sensor,8
vazao_setpoint_l_min,0
pressao_linha_bar,14
temperatura_produto_c,13
abertura_valvula_pct,12


In [ ]:
# Conte as linhas duplicadas
df.duplicated().sum()

np.int64(15)

In [ ]:
# Separe colunas numéricas e categóricas
colunas_numericas = df.select_dtypes(include = np.number).columns.to_list()
colunas_categoricas = df.select_dtypes(exclude = np.number).columns.to_list()

print("Numéricas:", colunas_numericas)
print("Categóricas:", colunas_categoricas)

Numéricas: ['vazao_setpoint_l_min', 'pressao_linha_bar', 'temperatura_produto_c', 'abertura_valvula_pct', 'vibracao_bomba_mm_s', 'ph_produto', 'consumo_energia_kwh', 'tempo_resposta_seg', 'erro_vazao_pct', 'leituras_invalidas', 'horas_operacao_bomba']
Categóricas: ['timestamp', 'linha_producao', 'turno', 'produto_quimico', 'tipo_valvula', 'status_sensor', 'status_instrumentacao']


## 4. Tratamento de duplicatas

**Tarefa:**  
Crie uma cópia da base e remova as linhas duplicadas.

In [ ]:
df_tratado = df.copy()

antes = len(df)
df_tratado = df.drop_duplicates()
depois = len(df_tratado)

print("Linhas antes:", antes)
print("Linhas depois:", depois)
print("Duplicatas removidas:", df.duplicated().sum())

Linhas antes: 235
Linhas depois: 220
Duplicatas removidas: 15


## 5. Tratamento de valores faltantes

**Tarefa:**  
- Use **mediana** nas colunas numéricas;
- Use **moda** nas colunas categóricas.

In [ ]:
# Preencha valores faltantes das colunas numéricas
for col in df_tratado.select_dtypes(include = ['number']).columns:
    df_tratado[col] = df_tratado[col].fillna(df_tratado[col].median())

# Preencha valores faltantes das colunas categóricas
for col in df_tratado.select_dtypes(include = ['object', 'category']):
    moda = df_tratado[col].mode()
    if not moda.empty:
        df_tratado[col] = df_tratado[col].fillna(moda[0])

print(df_tratado.isna().sum().sort_values(ascending=False))

timestamp                0
linha_producao           0
turno                    0
produto_quimico          0
tipo_valvula             0
status_sensor            0
vazao_setpoint_l_min     0
pressao_linha_bar        0
temperatura_produto_c    0
abertura_valvula_pct     0
vibracao_bomba_mm_s      0
ph_produto               0
consumo_energia_kwh      0
tempo_resposta_seg       0
erro_vazao_pct           0
leituras_invalidas       0
horas_operacao_bomba     0
status_instrumentacao    0
dtype: int64


/tmp/ipykernel_1669/666095922.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tratado[col] = df_tratado[col].fillna(df_tratado[col].median())
/tmp/ipykernel_1669/666095922.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tratado[col] = df_tratado[col].fillna(moda[0])


## 6. Tratamento de outliers com IQR

**Tarefa:**  
Implemente a função de tratamento de outliers por IQR e aplique na base.

In [ ]:
def tratar_outliers_iqr(df_entrada, colunas):
    df_saida = df_entrada.copy()
    relatorio = []

    for col in colunas:
        q1 = df_saida[col].quantile(0.25)
        q3 = df_saida[col].quantile(0.75)
        iqr = q3 - q1
        lim_inf = q1 - 1.5 * iqr
        lim_sup = q3 + 1.5 * iqr

        qtd_antes = ((df_saida[col] < lim_inf) | (df_saida[col] > lim_sup)).sum()

        df_saida[col] = np.where(df_saida[col] < lim_inf, lim_inf, df_saida[col])
        df_saida[col] = np.where(df_saida[col] > lim_sup, lim_sup, df_saida[col])

        qtd_depois = ((df_saida[col] < lim_inf) | (df_saida[col] > lim_sup)).sum()

        relatorio.append({
            'coluna': col,
            'limite_inferior': lim_inf,
            'limite_superior': lim_sup,
            'outliers_antes': int(qtd_antes),
            'outliers_depois': int(qtd_depois)
        })

    return df_saida, pd.DataFrame(relatorio)

In [ ]:
colunas_numericas_trat = df_tratado.select_dtypes(include=['number']).columns
df_tratado, relatorio_outliers = tratar_outliers_iqr(df_tratado, colunas_numericas_trat)

relatorio_outliers

,coluna,limite_inferior,limite_superior,outliers_antes,outliers_depois
0,vazao_setpoint_l_min,75.81375,168.10375,1,0
1,pressao_linha_bar,2.68500,6.98500,4,0
2,temperatura_produto_c,20.97375,56.08375,6,0
3,abertura_valvula_pct,16.14875,96.85875,7,0
4,vibracao_bomba_mm_s,0.37000,5.27000,3,0
5,ph_produto,3.64125,9.93125,6,0
6,consumo_energia_kwh,8.26500,39.84500,4,0
7,tempo_resposta_seg,0.41500,3.17500,2,0
8,erro_vazao_pct,-2.31875,9.53125,1,0
9,leituras_invalidas,-0.50000,3.50000,12,0


## 7. Codificação de variáveis categóricas

**Tarefa:**  
Separe `X` e `y` e aplique `get_dummies()` nas variáveis preditoras.

In [ ]:
if target and target in df_tratado.columns:
    X = df_tratado.drop(columns=[target])
    y = df_tratado[target]
else:
    X = df_tratado
    y = None

X_codificado = pd.get_dummies(X)

print("Dimensão antes:", X.shape)
print("Dimensão depois:", X_codificado.shape)
X_codificado.head()

Dimensão antes: (220, 17)
Dimensão depois: (220, 256)


,vazao_setpoint_l_min,pressao_linha_bar,temperatura_produto_c,abertura_valvula_pct,vibracao_bomba_mm_s,ph_produto,consumo_energia_kwh,tempo_resposta_seg,erro_vazao_pct,leituras_invalidas,horas_operacao_bomba,timestamp_2026-03-01 00:00:00,timestamp_2026-03-01 01:00:00,timestamp_2026-03-01 02:00:00,timestamp_2026-03-01 03:00:00,timestamp_2026-03-01 04:00:00,timestamp_2026-03-01 05:00:00,timestamp_2026-03-01 06:00:00,timestamp_2026-03-01 07:00:00,timestamp_2026-03-01 08:00:00,timestamp_2026-03-01 09:00:00,timestamp_2026-03-01 10:00:00,timestamp_2026-03-01 11:00:00,timestamp_2026-03-01 12:00:00,timestamp_2026-03-01 13:00:00,timestamp_2026-03-01 14:00:00,timestamp_2026-03-01 15:00:00,timestamp_2026-03-01 16:00:00,timestamp_2026-03-01 17:00:00,timestamp_2026-03-01 18:00:00,timestamp_2026-03-01 19:00:00,timestamp_2026-03-01 20:00:00,timestamp_2026-03-01 21:00:00,timestamp_2026-03-01 22:00:00,timestamp_2026-03-01 23:00:00,timestamp_2026-03-02 00:00:00,timestamp_2026-03-02 01:00:00,timestamp_2026-03-02 02:00:00,timestamp_2026-03-02 03:00:00,timestamp_2026-03-02 04:00:00,timestamp_2026-03-02 05:00:00,timestamp_2026-03-02 06:00:00,timestamp_2026-03-02 07:00:00,timestamp_2026-03-02 08:00:00,timestamp_2026-03-02 09:00:00,timestamp_2026-03-02 10:00:00,timestamp_2026-03-02 11:00:00,timestamp_2026-03-02 12:00:00,timestamp_2026-03-02 13:00:00,timestamp_2026-03-02 14:00:00,timestamp_2026-03-02 15:00:00,timestamp_2026-03-02 16:00:00,timestamp_2026-03-02 17:00:00,timestamp_2026-03-02 18:00:00,timestamp_2026-03-02 19:00:00,timestamp_2026-03-02 20:00:00,timestamp_2026-03-02 21:00:00,timestamp_2026-03-02 22:00:00,timestamp_2026-03-02 23:00:00,timestamp_2026-03-03 00:00:00,timestamp_2026-03-03 01:00:00,timestamp_2026-03-03 02:00:00,timestamp_2026-03-03 03:00:00,timestamp_2026-03-03 04:00:00,timestamp_2026-03-03 05:00:00,timestamp_2026-03-03 06:00:00,timestamp_2026-03-03 07:00:00,timestamp_2026-03-03 08:00:00,timestamp_2026-03-03 09:00:00,timestamp_2026-03-03 10:00:00,timestamp_2026-03-03 11:00:00,timestamp_2026-03-03 12:00:00,timestamp_2026-03-03 13:00:00,timestamp_2026-03-03 14:00:00,timestamp_2026-03-03 15:00:00,timestamp_2026-03-03 16:00:00,timestamp_2026-03-03 17:00:00,timestamp_2026-03-03 18:00:00,timestamp_2026-03-03 19:00:00,timestamp_2026-03-03 20:00:00,timestamp_2026-03-03 21:00:00,timestamp_2026-03-03 22:00:00,timestamp_2026-03-03 23:00:00,timestamp_2026-03-04 00:00:00,timestamp_2026-03-04 01:00:00,timestamp_2026-03-04 02:00:00,timestamp_2026-03-04 03:00:00,timestamp_2026-03-04 04:00:00,timestamp_2026-03-04 05:00:00,timestamp_2026-03-04 06:00:00,timestamp_2026-03-04 07:00:00,timestamp_2026-03-04 08:00:00,timestamp_2026-03-04 09:00:00,timestamp_2026-03-04 10:00:00,timestamp_2026-03-04 11:00:00,timestamp_2026-03-04 12:00:00,timestamp_2026-03-04 13:00:00,timestamp_2026-03-04 14:00:00,timestamp_2026-03-04 15:00:00,timestamp_2026-03-04 16:00:00,timestamp_2026-03-04 17:00:00,timestamp_2026-03-04 18:00:00,timestamp_2026-03-04 19:00:00,timestamp_2026-03-04 20:00:00,timestamp_2026-03-04 21:00:00,timestamp_2026-03-04 22:00:00,timestamp_2026-03-04 23:00:00,timestamp_2026-03-05 00:00:00,timestamp_2026-03-05 01:00:00,timestamp_2026-03-05 02:00:00,timestamp_2026-03-05 03:00:00,timestamp_2026-03-05 04:00:00,timestamp_2026-03-05 05:00:00,timestamp_2026-03-05 06:00:00,timestamp_2026-03-05 07:00:00,timestamp_2026-03-05 08:00:00,timestamp_2026-03-05 09:00:00,timestamp_2026-03-05 10:00:00,timestamp_2026-03-05 11:00:00,timestamp_2026-03-05 12:00:00,timestamp_2026-03-05 13:00:00,timestamp_2026-03-05 14:00:00,timestamp_2026-03-05 15:00:00,timestamp_2026-03-05 16:00:00,timestamp_2026-03-05 17:00:00,timestamp_2026-03-05 18:00:00,timestamp_2026-03-05 19:00:00,timestamp_2026-03-05 20:00:00,timestamp_2026-03-05 21:00:00,timestamp_2026-03-05 22:00:00,timestamp_2026-03-05 23:00:00,timestamp_2026-03-06 00:00:00,timestamp_2026-03-06 01:00:00,timestamp_2026-03-06 02:00:00,timestamp_2026-03-06 03:00:00,timestamp_2026-03-06 04:00:00,timestamp_2026-03-06 05:00:00,timestamp_20

## 8. Normalização e padronização

**Tarefa:**  
Crie:
- uma versão normalizada da base;
- uma versão padronizada da base.

In [ ]:
colunas_numericas_final = X_codificado.select_dtypes(include=['number']).columns

# Normalização
scaler_minmax = MinMaxScaler()
X_normalizado = X_codificado.copy()
X_normalizado[colunas_numericas_final] = scaler_minmax.fit_transform(X_codificado[colunas_numericas_final])

# Padronização
scaler_standard = StandardScaler()
X_padronizado = X_codificado.copy()
X_padronizado[colunas_numericas_final] = scaler_standard.fit_transform(X_codificado[colunas_numericas_final])

X_normalizado.head(), X_padronizado.head()

(   vazao_setpoint_l_min  pressao_linha_bar  temperatura_produto_c  \
 0              0.578214           0.412791               0.698839   
 1              0.623047           0.559302               0.417723   
 2              0.927063           0.552326               0.440224   
 3              0.909330           0.475581               0.533075   
 4              0.508176           0.491860               0.341961   
 
    abertura_valvula_pct  vibracao_bomba_mm_s  ph_produto  consumo_energia_kwh  \
 0              0.796323             0.644898    0.626192             0.308581   
 1              0.563267             0.253061    0.723172             0.548923   
 2              0.363415             0.742857    0.696145             0.689835   
 3              0.394019             0.416327    0.510135             0.209785   
 4              0.477032             0.328571    0.435413             0.213901   
 
    tempo_resposta_seg  erro_vazao_pct  leituras_invalidas  \
 0            1.000000

## 9. Base final pronta para modelagem

**Tarefa:**  
Escolha a base final que será utilizada em um futuro algoritmo de ML e reconecte a variável-alvo, se houver.

In [ ]:
base_final_modelagem = X_padronizado.copy()

if y is not None:
    base_final_modelagem[target] = y

print("Dimensão final:", base_final_modelagem.shape)
base_final_modelagem.head()

Dimensão final: (220, 257)


,vazao_setpoint_l_min,pressao_linha_bar,temperatura_produto_c,abertura_valvula_pct,vibracao_bomba_mm_s,ph_produto,consumo_energia_kwh,tempo_resposta_seg,erro_vazao_pct,leituras_invalidas,horas_operacao_bomba,timestamp_2026-03-01 00:00:00,timestamp_2026-03-01 01:00:00,timestamp_2026-03-01 02:00:00,timestamp_2026-03-01 03:00:00,timestamp_2026-03-01 04:00:00,timestamp_2026-03-01 05:00:00,timestamp_2026-03-01 06:00:00,timestamp_2026-03-01 07:00:00,timestamp_2026-03-01 08:00:00,timestamp_2026-03-01 09:00:00,timestamp_2026-03-01 10:00:00,timestamp_2026-03-01 11:00:00,timestamp_2026-03-01 12:00:00,timestamp_2026-03-01 13:00:00,timestamp_2026-03-01 14:00:00,timestamp_2026-03-01 15:00:00,timestamp_2026-03-01 16:00:00,timestamp_2026-03-01 17:00:00,timestamp_2026-03-01 18:00:00,timestamp_2026-03-01 19:00:00,timestamp_2026-03-01 20:00:00,timestamp_2026-03-01 21:00:00,timestamp_2026-03-01 22:00:00,timestamp_2026-03-01 23:00:00,timestamp_2026-03-02 00:00:00,timestamp_2026-03-02 01:00:00,timestamp_2026-03-02 02:00:00,timestamp_2026-03-02 03:00:00,timestamp_2026-03-02 04:00:00,timestamp_2026-03-02 05:00:00,timestamp_2026-03-02 06:00:00,timestamp_2026-03-02 07:00:00,timestamp_2026-03-02 08:00:00,timestamp_2026-03-02 09:00:00,timestamp_2026-03-02 10:00:00,timestamp_2026-03-02 11:00:00,timestamp_2026-03-02 12:00:00,timestamp_2026-03-02 13:00:00,timestamp_2026-03-02 14:00:00,timestamp_2026-03-02 15:00:00,timestamp_2026-03-02 16:00:00,timestamp_2026-03-02 17:00:00,timestamp_2026-03-02 18:00:00,timestamp_2026-03-02 19:00:00,timestamp_2026-03-02 20:00:00,timestamp_2026-03-02 21:00:00,timestamp_2026-03-02 22:00:00,timestamp_2026-03-02 23:00:00,timestamp_2026-03-03 00:00:00,timestamp_2026-03-03 01:00:00,timestamp_2026-03-03 02:00:00,timestamp_2026-03-03 03:00:00,timestamp_2026-03-03 04:00:00,timestamp_2026-03-03 05:00:00,timestamp_2026-03-03 06:00:00,timestamp_2026-03-03 07:00:00,timestamp_2026-03-03 08:00:00,timestamp_2026-03-03 09:00:00,timestamp_2026-03-03 10:00:00,timestamp_2026-03-03 11:00:00,timestamp_2026-03-03 12:00:00,timestamp_2026-03-03 13:00:00,timestamp_2026-03-03 14:00:00,timestamp_2026-03-03 15:00:00,timestamp_2026-03-03 16:00:00,timestamp_2026-03-03 17:00:00,timestamp_2026-03-03 18:00:00,timestamp_2026-03-03 19:00:00,timestamp_2026-03-03 20:00:00,timestamp_2026-03-03 21:00:00,timestamp_2026-03-03 22:00:00,timestamp_2026-03-03 23:00:00,timestamp_2026-03-04 00:00:00,timestamp_2026-03-04 01:00:00,timestamp_2026-03-04 02:00:00,timestamp_2026-03-04 03:00:00,timestamp_2026-03-04 04:00:00,timestamp_2026-03-04 05:00:00,timestamp_2026-03-04 06:00:00,timestamp_2026-03-04 07:00:00,timestamp_2026-03-04 08:00:00,timestamp_2026-03-04 09:00:00,timestamp_2026-03-04 10:00:00,timestamp_2026-03-04 11:00:00,timestamp_2026-03-04 12:00:00,timestamp_2026-03-04 13:00:00,timestamp_2026-03-04 14:00:00,timestamp_2026-03-04 15:00:00,timestamp_2026-03-04 16:00:00,timestamp_2026-03-04 17:00:00,timestamp_2026-03-04 18:00:00,timestamp_2026-03-04 19:00:00,timestamp_2026-03-04 20:00:00,timestamp_2026-03-04 21:00:00,timestamp_2026-03-04 22:00:00,timestamp_2026-03-04 23:00:00,timestamp_2026-03-05 00:00:00,timestamp_2026-03-05 01:00:00,timestamp_2026-03-05 02:00:00,timestamp_2026-03-05 03:00:00,timestamp_2026-03-05 04:00:00,timestamp_2026-03-05 05:00:00,timestamp_2026-03-05 06:00:00,timestamp_2026-03-05 07:00:00,timestamp_2026-03-05 08:00:00,timestamp_2026-03-05 09:00:00,timestamp_2026-03-05 10:00:00,timestamp_2026-03-05 11:00:00,timestamp_2026-03-05 12:00:00,timestamp_2026-03-05 13:00:00,timestamp_2026-03-05 14:00:00,timestamp_2026-03-05 15:00:00,timestamp_2026-03-05 16:00:00,timestamp_2026-03-05 17:00:00,timestamp_2026-03-05 18:00:00,timestamp_2026-03-05 19:00:00,timestamp_2026-03-05 20:00:00,timestamp_2026-03-05 21:00:00,timestamp_2026-03-05 22:00:00,timestamp_2026-03-05 23:00:00,timestamp_2026-03-06 00:00:00,timestamp_2026-03-06 01:00:00,timestamp_2026-03-06 02:00:00,timestamp_2026-03-06 03:00:00,timestamp_2026-03-06 04:00:00,timestamp_2026-03-06 05:00:00,timestamp_20

## 10. Resumo técnico

Preencha abaixo um resumo do que foi feito no seu pré-processamento.

In [ ]:
resumo = {
    'arquivo_utilizado': CAMINHO_ARQUIVO,
    'linhas_iniciais': df.shape[0],
    'colunas_iniciais': df.shape[1],
    'duplicatas_iniciais': df.duplicated().sum(),
    'linhas_finais': base_final_modelagem.shape[0],
    'colunas_finais': base_final_modelagem.shape[1],
    'target': target,
    'tratamento_faltantes': 'Preenchimento com Mediana (numéricas) e Moda (categóricas)',
    'tratamento_outliers': 'Método IQR com Winsorização',
    'codificacao': 'One-Hot Encoding usando pd.get_dummies',
    'transformacao_final': 'Padronização (StandardScaler)'
}

pd.DataFrame([resumo])

,arquivo_utilizado,linhas_iniciais,colunas_iniciais,duplicatas_iniciais,linhas_finais,colunas_finais,target,tratamento_faltantes,tratamento_outliers,codificacao,transformacao_final
0,/content/dados_desafio_preprocessamento_indust...,235,18,15,220,257,status_instrumentacao,Preenchimento com Mediana (numéricas) e Moda (...,Método IQR com Winsorização,One-Hot Encoding usando pd.get_dummies,Padronização (StandardScaler)


In [ ]:
# Exporte sua base final
base_final_modelagem.to_csv('base_final_desafio_preprocessamento.csv', index=False)
print('Arquivo salvo com sucesso.')

Arquivo salvo com sucesso.
